# Transformations Numériques des Variables ResStock
Entrée : `metadata_clean.parquet` → Sortie : `metadata_features.parquet`

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')
print('Shape original :', df.shape)
df_feat = df.copy()

Shape original : (549971, 391)


## 1. Isolation — R-values : `"R-11"` → `11.0`

In [2]:

# ── Insulation Wall → R-value d'assemblage (Table 45 du guide ResStock) ──────
# La R-value d'assemblage inclut l'ossature (ponts thermiques) + l'isolant ajouté
WALL_ASSEMBLY_R = {
    'Wood Stud, Uninsulated':           3.4,
    'Wood Stud, R-7':                   8.7,
    'Wood Stud, R-11':                 10.3,
    'Wood Stud, R-15':                 12.1,
    'Wood Stud, R-19':                 15.4,
    'CMU, 6-in Hollow, Uninsulated':    4.0,
    'CMU, 6-in Hollow, R-7':            9.4,
    'CMU, 6-in Hollow, R-11':          12.4,
    'CMU, 6-in Hollow, R-15':          15.0,
    'CMU, 6-in Hollow, R-19':          17.4,
    'Brick, 12-in, 3-wythe, Uninsulated': 4.9,
    'Brick, 12-in, 3-wythe, R-7':      10.3,
    'Brick, 12-in, 3-wythe, R-11':     13.3,
    'Brick, 12-in, 3-wythe, R-15':     15.9,
    'Brick, 12-in, 3-wythe, R-19':     18.3,
}
if 'in.insulation_wall' in df_feat.columns:
    df_feat['in.insulation_wall'] = df_feat['in.insulation_wall'].map(WALL_ASSEMBLY_R).astype('float32')
    print('insulation_wall → R assemblage OK', df_feat['in.insulation_wall'].value_counts().sort_index().to_dict())

# ── Insulation Ceiling → R-value d'assemblage (Table 53) ─────────────────────
CEILING_ASSEMBLY_R = {
    'None':        np.nan,   # pas de combles → NaN logique
    'Uninsulated':  2.1,
    'R-7':          8.7,
    'R-13':        14.6,
    'R-19':        20.6,
    'R-30':        31.6,
    'R-38':        39.6,
    'R-49':        50.6,
}
if 'in.insulation_ceiling' in df_feat.columns:
    df_feat['in.insulation_ceiling'] = df_feat['in.insulation_ceiling'].map(CEILING_ASSEMBLY_R).astype('float32')
    print('insulation_ceiling → R assemblage OK')

# ── Insulation Roof → R-value d'assemblage (Table 47) ────────────────────────
ROOF_ASSEMBLY_R = {
    'Unfinished, Uninsulated':  2.3,
    'Finished, Uninsulated':    3.7,
    'Finished, R-7':           10.2,
    'Finished, R-13':          14.3,
    'Finished, R-19':          21.2,
    'Finished, R-30':          29.7,
    'Finished, R-38':          36.5,
    'Finished, R-49':          47.0,
}
if 'in.insulation_roof' in df_feat.columns:
    df_feat['in.insulation_roof'] = df_feat['in.insulation_roof'].map(ROOF_ASSEMBLY_R).astype('float32')
    print('insulation_roof → R assemblage OK', df_feat['in.insulation_roof'].value_counts().sort_index().to_dict())

# ── Insulation Floor → R-value d'assemblage (Table 55) ───────────────────────
FLOOR_ASSEMBLY_R = {
    'None':          np.nan,   # dalle directe → pas de plancher isolé
    'Uninsulated':    5.3,
    'Ceiling R-13':  17.8,
    'Ceiling R-19':  22.6,
    'Ceiling R-30':  30.3,
}
if 'in.insulation_floor' in df_feat.columns:
    df_feat['in.insulation_floor'] = df_feat['in.insulation_floor'].map(FLOOR_ASSEMBLY_R).astype('float32')
    print('insulation_floor → R assemblage OK')

# ── Insulation Foundation Wall → R nominale (Table 59) ───────────────────────
FOUNDATION_WALL_R = {
    'None':                np.nan,
    'Uninsulated':          0.0,
    'Wall R-5, Exterior':   5.0,
    'Wall R-10, Exterior': 10.0,
    'Wall R-15, Exterior': 15.0,
}
if 'in.insulation_foundation_wall' in df_feat.columns:
    df_feat['in.insulation_foundation_wall'] = df_feat['in.insulation_foundation_wall'].map(FOUNDATION_WALL_R).astype('float32')
    print('insulation_foundation_wall → R OK')

# ── Insulation Rim Joist → R nominale (Table 61, = même valeur que foundation wall) ─
RIM_JOIST_R = {
    'None':          np.nan,
    'Uninsulated':    0.0,
    'R-5, Exterior':  5.0,
    'R-10, Exterior': 10.0,
    'R-15, Exterior': 15.0,
}
if 'in.insulation_rim_joist' in df_feat.columns:
    df_feat['in.insulation_rim_joist'] = df_feat['in.insulation_rim_joist'].map(RIM_JOIST_R).astype('float32')
    print('insulation_rim_joist → R OK')

# ── Insulation Slab → 2 colonnes : périmètre vertical + sous-dalle horizontal (Table 57) ──
# La position de l'isolation a un impact thermique différent
SLAB_MAP = {
    'None':                      (0.0,  0.0),
    'Uninsulated':               (0.0,  0.0),
    '2ft R5 Under, Horizontal':  (0.0,  5.0),   # sous-dalle
    '2ft R10 Under, Horizontal': (0.0, 10.0),   # sous-dalle
    '4ft R5 Under, Horizontal':  (0.0,  5.0),   # sous-dalle (plus large)
    '2ft R5 Perimeter, Vertical':(5.0,  0.0),   # périmètre
    '2ft R10 Perimeter, Vertical':(10.0, 0.0),  # périmètre
    'R10 Whole Slab, Horizontal':(0.0, 10.0),   # toute la surface
}
if 'in.insulation_slab' in df_feat.columns:
    slab_parsed = df_feat['in.insulation_slab'].apply(
        lambda v: pd.Series(SLAB_MAP.get(str(v), (np.nan, np.nan)),
                            index=['in.slab_perimeter_r', 'in.slab_under_r'])
    ).astype('float32')
    df_feat = pd.concat([df_feat.drop(columns=['in.insulation_slab']), slab_parsed], axis=1)
    print('insulation_slab → in.slab_perimeter_r + in.slab_under_r OK')

print(f'\nShape : {df_feat.shape}')

INSUL_COLS = [
    'in.insulation_wall', 'in.insulation_ceiling', 'in.insulation_roof',
    'in.insulation_floor', 'in.insulation_foundation_wall', 'in.insulation_rim_joist',
    'in.slab_perimeter_r', 'in.slab_under_r',
]
INSUL_COLS = [c for c in INSUL_COLS if c in df_feat.columns]
print(f'INSUL_COLS defines: {INSUL_COLS}')


insulation_wall → R assemblage OK {3.4000000953674316: 167608, 4.0: 7707, 4.900000095367432: 54176, 8.699999809265137: 40407, 9.399999618530273: 2227, 10.300000190734863: 139342, 12.100000381469727: 26059, 12.399999618530273: 5021, 13.300000190734863: 30332, 15.0: 915, 15.399999618530273: 57685, 15.899999618530273: 7746, 17.399999618530273: 2396, 18.299999237060547: 8350}
insulation_ceiling → R assemblage OK
insulation_roof → R assemblage OK {2.299999952316284: 235995, 3.700000047683716: 10384, 10.199999809265137: 20178, 14.300000190734863: 39487, 21.200000762939453: 58591, 29.700000762939453: 103846, 36.5: 61405, 47.0: 20085}
insulation_floor → R assemblage OK
insulation_foundation_wall → R OK


insulation_rim_joist → R OK


insulation_slab → in.slab_perimeter_r + in.slab_under_r OK

Shape : (549971, 392)
INSUL_COLS defines: ['in.insulation_wall', 'in.insulation_ceiling', 'in.insulation_roof', 'in.insulation_floor', 'in.insulation_foundation_wall', 'in.insulation_rim_joist', 'in.slab_perimeter_r', 'in.slab_under_r']


## 2. Surface habitable — midpoint ft² → m² : `"1000-1499"` → `116.1 m²`

In [3]:
def midpoint_m2(val):
    if pd.isna(val): return np.nan
    s = str(val).replace(',', '')
    if '+' in s:
        return float(s.replace('+', '')) * 0.0929
    m = re.match(r'(\d+)-(\d+)', s)
    if m:
        midpoint = (float(m.group(1)) + float(m.group(2))) / 2
        return round(midpoint * 0.0929, 1)
    return np.nan

if 'in.geometry_floor_area' in df_feat.columns:
    df_feat['in.geometry_floor_area'] = df_feat['in.geometry_floor_area'].apply(midpoint_m2)

print('Surface (m2) — stats :')
print(df_feat['in.geometry_floor_area'].describe().round(1))

Surface (m2) — stats :
count    549971.0
mean        153.1
std          82.4
min          23.2
25%          81.2
50%         116.1
75%         209.0
max         371.6
Name: in.geometry_floor_area, dtype: float64


## 3. Consignes de température — °F → °C : `"68F"` → `20.0°C`

In [4]:
def f_to_c(val):
    if pd.isna(val): return np.nan
    m = re.search(r'(\d+\.?\d*)', str(val))
    return round((float(m.group(1)) - 32) * 5 / 9, 1) if m else np.nan

SP_COLS = [
    'in.heating_setpoint', 'in.cooling_setpoint',
    'in.heating_setpoint_offset_magnitude', 'in.cooling_setpoint_offset_magnitude',
]
for col in SP_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(f_to_c)

print('Consigne chauffage (degC) :', df_feat['in.heating_setpoint'].value_counts().head())
print('Consigne clim (degC)      :', df_feat['in.cooling_setpoint'].value_counts().head())

Consigne chauffage (degC) : in.heating_setpoint
21.1    127983
20.0    112473
22.2     85121
12.8     66637
23.9     51949
Name: count, dtype: int64
Consigne clim (degC)      : in.cooling_setpoint
22.2    109045
21.1    104648
23.9    103578
20.0     63600
24.4     43812
Name: count, dtype: int64


## 4. Efficacité HVAC — `"AFUE, 80%"` → `0.80` | `"SEER, 13"` → `13.0`

In [5]:
def extract_efficiency(val):
    if pd.isna(val): return np.nan
    s = str(val)
    # AFUE / percentage (e.g. "80% AFUE", "100% Efficiency")
    m_pct = re.search(r'(\d+\.?\d*)%', s)
    if m_pct: return round(float(m_pct.group(1)) / 100, 3)
    # HSPF — heating metric for heat pumps (e.g. "ASHP, SEER 10, 6.2 HSPF")
    m_hspf = re.search(r'(\d+\.?\d*)\s*HSPF', s)
    if m_hspf: return float(m_hspf.group(1))
    # SEER / EER — number at end of string (e.g. "AC, SEER 13", "Room AC, EER 10.7")
    m_num = re.search(r'[\s,]+(\d+\.?\d*)\s*$', s)
    if m_num: return float(m_num.group(1))
    return np.nan

EFF_COLS = ['in.hvac_heating_efficiency', 'in.hvac_cooling_efficiency']
for col in EFF_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(extract_efficiency)

print('Heating efficiency :')
print(df_feat['in.hvac_heating_efficiency'].value_counts().sort_index())
print('\nCooling efficiency :')
print(df_feat['in.hvac_cooling_efficiency'].value_counts().sort_index())

Heating efficiency :
in.hvac_heating_efficiency
0.600      17485
0.680      15214
0.760      19159
0.800     154028
0.900       2483
0.925      85157
1.000     107073
6.200       4075
7.700      41979
8.200       5317
8.500      41083
14.000        83
Name: count, dtype: int64

Cooling efficiency :
in.hvac_cooling_efficiency
8.0       4925
8.5       2332
9.8      14047
10.0     32991
10.7     52419
12.0     40194
13.0    159099
15.0     73338
Name: count, dtype: int64


## 5. Variables ordinales simples — string → int

In [6]:
NUM_COLS = [
    'in.geometry_stories',
    'in.bedrooms',
    'in.occupants',
    'in.air_leakage_to_outside_ach50',
]
for col in NUM_COLS:
    if col in df_feat.columns:
        df_feat[col] = pd.to_numeric(df_feat[col], errors='coerce')

print('Stories   :', df_feat['in.geometry_stories'].value_counts().to_dict())
print('Occupants :', df_feat['in.occupants'].value_counts().sort_index().to_dict())

Stories   : {1: 292940, 2: 183548, 3: 42412, 4: 10864, 5: 4309, 6: 3801, 21: 3590, 20: 1145, 12: 1025, 10: 898, 8: 893, 7: 831, 9: 725, 13: 671, 15: 628, 35: 611, 14: 583, 11: 497}
Occupants : {0.0: 66637, 1.0: 132575, 2.0: 167006, 3.0: 75609, 4.0: 61637, 5.0: 28865, 6.0: 11193, 7.0: 3943, 8.0: 1482, 9.0: 512}


## 6. Revenu — midpoint : `"10000-14999"` → `12500`

In [7]:
def midpoint_income(val):
    if pd.isna(val): return np.nan
    s = str(val).replace(',', '').replace('$', '').strip()
    if s.startswith('<'):
        return float(s[1:]) / 2
    if s.endswith('+'):
        return float(s[:-1])
    m = re.match(r'(\d+)-(\d+)', s)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    return pd.to_numeric(s, errors='coerce')

if 'in.income' in df_feat.columns:
    df_feat['in.income'] = df_feat['in.income'].apply(midpoint_income)

print('Revenu (USD) — stats :')
print(df_feat['in.income'].describe().round(0))

Revenu (USD) — stats :
count    483334.0
mean      77435.0
std       57541.0
min        5000.0
25%       32500.0
50%       65000.0
75%      110000.0
max      200000.0
Name: in.income, dtype: float64


## 7. Vintage → âge du bâtiment : `"1980s"` → `44` ans

In [8]:
REFERENCE_YEAR = 2024

def vintage_to_age(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s.startswith('<'):
        decade_mid = 1930
    else:
        m = re.match(r'(\d{4})s?', s)
        if not m:
            return np.nan
        decade_mid = int(m.group(1)) + 5
    return REFERENCE_YEAR - decade_mid

if 'in.vintage' in df_feat.columns:
    df_feat['in.vintage'] = df_feat['in.vintage'].apply(vintage_to_age)
    print('Âge du bâtiment (années) — stats :')
    print(df_feat['in.vintage'].describe().round(1))

Âge du bâtiment (années) — stats :
count    549971.0
mean         49.0
std          25.0
min           9.0
25%          29.0
50%          49.0
75%          69.0
max          94.0
Name: in.vintage, dtype: float64


## 8. Variables binaires Yes/No → 0 / 1

In [9]:
bool_cols = [
    c for c in df_feat.columns
    if set(df_feat[c].dropna().astype(str).str.strip().unique()) <= {'Yes', 'No'}
]

for c in bool_cols:
    df_feat[c] = df_feat[c].map({'Yes': 1, 'No': 0}).astype('Int8')

zero_var = [c for c in bool_cols if df_feat[c].nunique(dropna=True) <= 1]

print(f'Colonnes binaires converties ({len(bool_cols)}) :')
for c in bool_cols:
    pct_yes = df_feat[c].mean() * 100
    flag = '  ← variance nulle' if c in zero_var else ''
    print(f'  {c:<55} {pct_yes:5.1f}% Yes{flag}')
print("\nColonnes modifiées :")
print(bool_cols)


Colonnes binaires converties (11) :
  in.aiannh_area                                            1.6% Yes
  in.cooling_setpoint_has_offset                           35.8% Yes
  in.electric_vehicle_outlet_access                        56.5% Yes
  in.electric_vehicle_ownership                             1.1% Yes
  in.has_pv                                                 1.0% Yes
  in.heating_setpoint_has_offset                           43.8% Yes
  in.hvac_has_ducts                                        77.0% Yes
  in.hvac_has_zonal_electric_heating                        6.4% Yes
  in.hvac_system_is_faulted                                 0.0% Yes  ← variance nulle
  in.hvac_system_is_scaled                                  0.0% Yes  ← variance nulle
  in.water_heater_in_unit                                  85.7% Yes

Colonnes modifiées :
['in.aiannh_area', 'in.cooling_setpoint_has_offset', 'in.electric_vehicle_outlet_access', 'in.electric_vehicle_ownership', 'in.has_pv', 'in.heating

## 9. Heures de ventilation — `"Hour12"` → `12`

In [10]:
import re

def extract_hour(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r'Hour(\d+)', str(val))
    return int(m.group(1)) if m else np.nan

HOUR_COLS = ['in.bathroom_spot_vent_hour', 'in.range_spot_vent_hour']
for col in HOUR_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(extract_hour)

print('Heures ventilation :', df_feat['in.bathroom_spot_vent_hour'].value_counts().head())
print('Heures ventilation :', df_feat['in.range_spot_vent_hour'].value_counts().head())

Heures ventilation : in.bathroom_spot_vent_hour
2     33350
6     33350
22    33350
0     33349
1     33349
Name: count, dtype: int64
Heures ventilation : in.range_spot_vent_hour
17    82250
18    64154
16    50446
19    32901
12    31255
Name: count, dtype: int64


## 10. Niveaux d'utilisation — "120%" → 1.2

In [11]:
'''
def extract_pct(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r'(\d+)', str(val))
    return int(m.group(1)) / 100 if m else np.nan

USAGE_COLS = [
    'in.clothes_dryer_usage_level',
    'in.clothes_washer_usage_level',
    'in.dishwasher_usage_level',
    'in.cooking_range_usage_level',
]
for col in USAGE_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(extract_pct)

print('Niveaux d\'utilisation :', df_feat['in.clothes_washer_usage_level'].value_counts().head())

'''

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_11000\11873992.py:1: SyntaxWarning: invalid escape sequence '\d'
  '''


"\ndef extract_pct(val):\n    if pd.isna(val):\n        return np.nan\n    m = re.search(r'(\\d+)', str(val))\n    return int(m.group(1)) / 100 if m else np.nan\n\nUSAGE_COLS = [\n    'in.clothes_dryer_usage_level',\n    'in.clothes_washer_usage_level',\n    'in.dishwasher_usage_level',\n    'in.cooking_range_usage_level',\n]\nfor col in USAGE_COLS:\n    if col in df_feat.columns:\n        df_feat[col] = df_feat[col].apply(extract_pct)\n\nprint('Niveaux d'utilisation :', df_feat['in.clothes_washer_usage_level'].value_counts().head())\n\n"

## 11. Convertir des porcentage 120% ==>1.2 : 

In [12]:

def pourcentage(pr):
    if pd.isna(pr):
        return np.nan
    p=re.search(r'(\d+)\s*%', str(pr))
    return int(p.group(1))/100 if p else np.nan


COLS_POURCENTAGE = [ 'in.clothes_dryer_usage_level','in.clothes_washer_usage_level',
                              'in.dishwasher_usage_level','in.cooking_range_usage_level',
                              'in.hot_water_fixtures','in.lighting_interior_use','in.lighting_other_use',
                              'in.refrigerator_usage_level',
                              'in.plug_load_diversity',
                              'in.plug_loads'
                              ]
for col in COLS_POURCENTAGE:
    print(col)
    print(df_feat[col].value_counts(dropna=False).head(10))
for col in COLS_POURCENTAGE:
    df_feat[col] = df_feat[col].apply(pourcentage)

print(df_feat[COLS_POURCENTAGE].head())
print(df_feat[COLS_POURCENTAGE].dtypes)

in.clothes_dryer_usage_level
in.clothes_dryer_usage_level
100% Usage    274985
80% Usage     137495
120% Usage    137491
Name: count, dtype: int64
in.clothes_washer_usage_level
in.clothes_washer_usage_level
100% Usage    274985
80% Usage     137495
120% Usage    137491
Name: count, dtype: int64
in.dishwasher_usage_level
in.dishwasher_usage_level
100% Usage    274985
80% Usage     137495
120% Usage    137491
Name: count, dtype: int64
in.cooking_range_usage_level
in.cooking_range_usage_level
100% Usage    274985
80% Usage     137495
120% Usage    137491
Name: count, dtype: int64
in.hot_water_fixtures
in.hot_water_fixtures
100% Usage    100126
90% Usage      92136
80% Usage      76733
110% Usage     73879
70% Usage      54487
120% Usage     48777
130% Usage     37367
140% Usage     20825
60% Usage      19968
150% Usage     11411
Name: count, dtype: int64
in.lighting_interior_use
in.lighting_interior_use
100% Usage    549971
Name: count, dtype: int64
in.lighting_other_use
in.lighting_other

   in.clothes_dryer_usage_level  in.clothes_washer_usage_level  \
0                           1.0                            1.0   
1                           1.2                            1.2   
2                           1.2                            1.2   
3                           1.2                            1.2   
4                           0.8                            0.8   

   in.dishwasher_usage_level  in.cooking_range_usage_level  \
0                        1.0                           1.0   
1                        1.2                           1.2   
2                        1.2                           1.2   
3                        1.2                           1.2   
4                        0.8                           0.8   

   in.hot_water_fixtures  in.lighting_interior_use  in.lighting_other_use  \
0                    1.1                       1.0                    1.0   
1                    1.1                       1.0                    1.0   

## 12. Convesion des colonnes : 'in.cooling_unavailable_days' 
#### (ex: "Never" → 0  ,"1 week" → 7 )


In [13]:
def unavailable_days(days_str):
    if pd.isna(days_str):
        return np.nan
    s = str(days_str).strip().lower()
    if s == 'never':       return 0
    if s == 'year round':  return 365
    m_month = re.search(r'(\d+)\s*month', s)
    if m_month: return int(m_month.group(1)) * 30
    m_week  = re.search(r'(\d+)\s*week',  s)
    if m_week:  return int(m_week.group(1)) * 7
    m_day   = re.search(r'(\d+)\s*day',   s)
    if m_day:   return int(m_day.group(1))
    return np.nan

COLS_UNAVAILABLE_DAYS = ['in.cooling_unavailable_days', 'in.heating_unavailable_days']

for col in COLS_UNAVAILABLE_DAYS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(unavailable_days).astype('float32')
        # NaN → médiane (cas rares)
        median_val = df_feat[col].median()
        df_feat[col] = df_feat[col].fillna(median_val)
        print(f'{col} → float32 ✓  (médiane={median_val:.0f} jours)')

in.cooling_unavailable_days → float32 ✓  (médiane=0 jours)
in.heating_unavailable_days → float32 ✓  (médiane=0 jours)


## 13. Conversion de 'in.duct_leakage_and_insulation' 
##### on extrait 2 feature le leakage (%) et l-isolation (R-value -> int)

In [14]:

def duct_leakage_pourcentage(val):

    if pd.isna(val) or str(val).strip().lower() == "none":
        return np.nan
    m = re.search(r'(\d+(?:\.\d+)?)\s*%', str(val))

    return float(m.group(1)) / 100 if m else np.nan


def duct_insulation(val):
    if pd.isna(val) or str(val).strip().lower() == "none":
        return 0
    s = str(val).lower()

    if "r-4" in s:
        return 4

    if "r-6" in s:
        return 6

    if "r-8" in s:
        return 8

    if "uninsulated" in s:
        return 0

    return np.nan

COLS_DUCT_LEAKAGE_AND_INSULATION = ['in.duct_leakage_and_insulation']
df_feat['in.duct_leakage'] = df_feat['in.duct_leakage_and_insulation'].apply(duct_leakage_pourcentage)
df_feat['in.duct_insulation'] = df_feat['in.duct_leakage_and_insulation'].apply(duct_insulation)

## 14. Conversion de la localisation des ducts : 'in.duct_location'

In [15]:
def duct_location(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()

    # Mapping exact (evite le bug 'heated basement' in 'unheated basement')
    DUCT_LOC_MAP = {
        'None':               0,  # pas de gaines
        'Living Space':       1,  # espace conditionne, pertes nulles
        'Heated Basement':    2,  # conditionne, pertes minimes
        'Unheated Basement':  3,  # non conditionne, pertes moderees
        'Crawlspace':         4,  # non conditionne, pertes moderees-elevees
        'Garage':             5,  # non conditionne, pertes elevees
        'Attic':              6,  # non conditionne, pire cas (chaleur ete)
    }
    return DUCT_LOC_MAP.get(s, np.nan)


df_feat['in.duct_location_int'] = df_feat['in.duct_location'].apply(duct_location)
print('duct_location_int corrige :')
print(df_feat.groupby(df_feat['in.duct_location'])['in.duct_location_int'].first().sort_values())


duct_location_int corrige :
in.duct_location
None                 0
Living Space         1
Heated Basement      2
Unheated Basement    3
Crawlspace           4
Garage               5
Attic                6
Name: in.duct_location_int, dtype: int64


## 15. Conversion de : 'in.eaves' de 2f^t -> m

In [16]:
'''def convert_to_meters(ft_str):
    if pd.isna(ft_str):
        return np.nan

    m = re.search(r'(\d+\.?\d*)', str(ft_str))
    if m:
        feet = float(m.group(1))
        meters = feet * 0.3048
        return meters

    return np.nan

LENGHT_COLS = ['in.eaves', 'in.sqft..ft2']
for col in LENGHT_COLS:
    if col in df_feat.columns:
            df_feat[col] = df_feat[col].apply(convert_to_meters)




#df_feat['in.eaves_length'] = df_feat['in.eaves'].apply(convert_to_meters)
'''

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_11000\2263912219.py:1: SyntaxWarning: invalid escape sequence '\d'
  '''def convert_to_meters(ft_str):


"def convert_to_meters(ft_str):\n    if pd.isna(ft_str):\n        return np.nan\n\n    m = re.search(r'(\\d+\\.?\\d*)', str(ft_str))\n    if m:\n        feet = float(m.group(1))\n        meters = feet * 0.3048\n        return meters\n\n    return np.nan\n\nLENGHT_COLS = ['in.eaves', 'in.sqft..ft2']\nfor col in LENGHT_COLS:\n    if col in df_feat.columns:\n            df_feat[col] = df_feat[col].apply(convert_to_meters)\n\n\n\n\n#df_feat['in.eaves_length'] = df_feat['in.eaves'].apply(convert_to_meters)\n"

## 16. Résumé — colonnes transformées

In [17]:
transformed = (
    INSUL_COLS
    + ['in.geometry_floor_area']
    + SP_COLS
    + EFF_COLS
    + NUM_COLS
    + ['in.income', 'in.vintage']
    + bool_cols
    + HOUR_COLS
    + COLS_POURCENTAGE
    + COLS_UNAVAILABLE_DAYS
    + ['in.duct_leakage', 'in.duct_insulation']
    + ['in.duct_location_int']
    + ['in.eaves_length']   
)
transformed = [c for c in transformed if c in df_feat.columns]

print(f'Colonnes transformées : {len(transformed)}\n')
print(f'{"Colonne":<55} {"Type original":>15} {"Type final":>12} {"% NaN final":>12}')
print('-' * 100)
for col in transformed:
    t_orig  = str(df[col].dtype) if col in df.columns else 'new'
    t_new   = str(df_feat[col].dtype)
    pct_nan = df_feat[col].isna().mean() * 100
    print(f'{col:<55} {t_orig:>15} {t_new:>12} {pct_nan:>11.1f}%')

Colonnes transformées : 49

Colonne                                                   Type original   Type final  % NaN final
----------------------------------------------------------------------------------------------------
in.insulation_wall                                                  str      float32         0.0%
in.insulation_ceiling                                               str      float32        57.1%
in.insulation_roof                                                  str      float32         0.0%
in.insulation_floor                                                 str      float32        39.4%
in.insulation_foundation_wall                                       str      float32        48.2%
in.insulation_rim_joist                                             str      float32        48.2%
in.slab_perimeter_r                                                 new      float32         0.0%
in.slab_under_r                                                     new      float32   

## 17. Sauvegarde → `metadata_features.parquet`

In [18]:
out_path = DATA_PROCESSED / 'metadata_features.parquet'
df_feat.to_parquet(out_path, index=False)
print(f'Sauvegarde : {out_path}')
print(f'Shape final : {df_feat.shape}')

Sauvegarde : C:\Users\bamdyoun\stage\FlexiMax\data\processed\metadata_features.parquet
Shape final : (549971, 395)
